# Plot DJF ENSO Cross-Section Correlations

Load cached longitude-pressure and latitude-pressure DJF Niño3.4 correlations only, then render individual single-panel figures for q-shading and w-shading cross-sections.

# 1. Setup

In [ ]:
from pathlib import Path

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.ticker import FuncFormatter

plt.rcParams.update({
    'font.family': 'Helvetica',
    'font.sans-serif': ['Helvetica', 'Arial', 'DejaVu Sans'],
    'axes.titleweight': 'regular',
    'figure.dpi': 120,
})

PROJECT_ROOT = Path('/Users/rizzie/TugasAkhir/data_processing')
CACHE_DIR = PROJECT_ROOT / 'data/intermediate/divided_correlation'
RESULTS_DIR = PROJECT_ROOT / 'results/analisis_korelasi_26-19'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

LON_CACHE = CACHE_DIR / 'corr_xsec_lon_pressure_q_u_omega.nc'
LAT_CACHE = CACHE_DIR / 'corr_xsec_lat_pressure_q_v_omega.nc'

LON_BAND = slice(95.0, 125.0)
LAT_BAND = slice(-10.0, 5.0)
CORR_LEVELS = np.arange(-1.0, 1.01, 0.05)
CORR_TICKS = np.arange(-1.0, 1.01, 0.25)
PRESSURE_TICKS = [1000, 925, 850, 700, 500, 400, 300, 200, 100]
LON_TICKS = np.array([95, 110, 125])
LAT_TICKS = np.array([-10, 0, 5])

QUIVER_SCALE = 14.0
QUIVER_X_TARGET = 12
QUIVER_Y_TARGET = 6


def quiver_step(size, target):
    return max(1, int(np.ceil(size / target)))


def format_lon(value, pos=None):
    return f'{value:.0f}E'


def format_lat(value, pos=None):
    if np.isclose(value, 0):
        return '0'
    return f"{abs(value):.0f}{'N' if value > 0 else 'S'}"


def load_cache(path):
    return xr.open_dataset(path).load()


def scale_vertical_component(horiz_values, omega_values):
    valid = np.isfinite(horiz_values) & np.isfinite(omega_values)
    if not valid.any():
        return omega_values, 1.0

    horiz_ref = float(np.nanpercentile(np.abs(horiz_values[valid]), 90))
    omega_ref = float(np.nanpercentile(np.abs(omega_values[valid]), 90))
    if not np.isfinite(horiz_ref) or horiz_ref == 0 or not np.isfinite(omega_ref) or omega_ref == 0:
        return omega_values, 1.0

    omega_scale = horiz_ref / omega_ref
    omega_scale = float(np.clip(omega_scale, 1.0, 100.0))
    return omega_values * omega_scale, omega_scale


def finite_count(da):
    return int(np.isfinite(da.values).sum())


def render_cross_section(
    shade_da,
    horiz_da,
    omega_da,
    *,
    x_dim,
    title,
    x_label,
    x_ticks,
    cbar_label,
    out_name,
    vector_label,
):
    fig, ax = plt.subplots(figsize=(10.5, 6.4))

    shade_2d = shade_da.transpose('level', x_dim)
    horiz_2d = horiz_da.transpose('level', x_dim)
    omega_2d = omega_da.transpose('level', x_dim)

    x = shade_2d[x_dim].values
    y = shade_2d['level'].values
    shade_values = shade_2d.values
    shade_has_data = np.isfinite(shade_values).any()

    if shade_has_data:
        mesh = ax.contourf(
            x,
            y,
            shade_values,
            levels=CORR_LEVELS,
            cmap='bwr',
            extend='both',
        )
        ax.contour(
            x,
            y,
            shade_values,
            levels=CORR_TICKS,
            colors='k',
            linewidths=0.5,
            alpha=0.45,
        )
    else:
        mesh = plt.cm.ScalarMappable(norm=Normalize(vmin=-1, vmax=1), cmap='bwr')
        mesh.set_array([])
        ax.text(
            0.5,
            0.55,
            'shading field unavailable / all NaN',
            transform=ax.transAxes,
            ha='center',
            va='center',
            fontsize=11,
            bbox={'facecolor': 'white', 'alpha': 0.85, 'edgecolor': '0.7'},
        )

    x_step = 1
    y_step = 1
    horiz_sub = horiz_2d.values[::y_step, ::x_step]
    omega_sub = omega_2d.values[::y_step, ::x_step]
    omega_plot, omega_scale = scale_vertical_component(horiz_sub, omega_sub)
    quiver_ok = np.isfinite(horiz_sub).any() and np.isfinite(omega_plot).any() and np.isfinite(horiz_sub - omega_plot).sum() >= 3

    if quiver_ok:
        q = ax.quiver(
            x[::x_step],
            y[::y_step],
            np.ma.masked_invalid(horiz_sub),
            np.ma.masked_invalid(-omega_plot),
            color='black',
            scale=QUIVER_SCALE,
            width=0.0020,
            headwidth=3.5,
            headlength=4.2,
            alpha=0.9,
            pivot='mid',
            zorder=4,
        )
        ax.quiverkey(
            q,
            X=0.78,
            Y=1.04,
            U=1,
            label='component r = 1.0',
            labelpos='E',
            coordinates='axes',
            color='black',
        )
    else:
        ax.text(
            0.02,
            0.04,
            'omega correlation unavailable / all NaN',
            transform=ax.transAxes,
            ha='left',
            va='bottom',
            fontsize=10,
            bbox={'facecolor': 'white', 'alpha': 0.85, 'edgecolor': '0.7'},
        )

    ax.text(
        0.01,
        1.01,
        f'{vector_label}; omega scaled x{omega_scale:.1f} for display, not a physical unit conversion',
        transform=ax.transAxes,
        ha='left',
        va='bottom',
        fontsize=9.5,
    )
    ax.set_title(title, loc='left', fontsize=14, fontweight='bold', pad=22)
    ax.set_xlabel(x_label, fontsize=13, labelpad=8)
    ax.set_ylabel('Pressure level (hPa)', fontsize=13, labelpad=10)
    ax.set_xticks(x_ticks)
    ax.set_yticks(PRESSURE_TICKS)
    ax.set_ylim(1000, 100)
    if x_dim == 'lon':
        ax.set_xlim(LON_BAND.start, LON_BAND.stop)
    else:
        ax.set_xlim(LAT_BAND.start, LAT_BAND.stop)
    ax.tick_params(axis='both', labelsize=10)
    ax.grid(True, linestyle='--', alpha=0.25, linewidth=0.5)
    ax.xaxis.set_major_formatter(FuncFormatter(format_lon if x_dim == 'lon' else format_lat))

    cbar = fig.colorbar(mesh, ax=ax, pad=0.04, shrink=0.92, ticks=CORR_TICKS)
    cbar.set_label(cbar_label, fontsize=12, labelpad=10)
    cbar.ax.tick_params(labelsize=10)

    fig.tight_layout(rect=[0, 0.02, 1, 0.95])
    fig.savefig(RESULTS_DIR / out_name, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(fig)


# 2. Load Cached Correlations

In [ ]:
lon_ds = load_cache(LON_CACHE)
lat_ds = load_cache(LAT_CACHE)

lon_q = {
    'p1': lon_ds['q_corr_p1'],
    'p2': lon_ds['q_corr_p2'],
    'delta': lon_ds['q_corr_delta'],
}
lon_u = {
    'p1': lon_ds['u_corr_p1'],
    'p2': lon_ds['u_corr_p2'],
    'delta': lon_ds['u_corr_delta'],
}
lon_omega = {
    'p1': lon_ds['omega_corr_p1'],
    'p2': lon_ds['omega_corr_p2'],
    'delta': lon_ds['omega_corr_delta'],
}

lat_q = {
    'p1': lat_ds['q_corr_p1'],
    'p2': lat_ds['q_corr_p2'],
    'delta': lat_ds['q_corr_delta'],
}
lat_v = {
    'p1': lat_ds['v_corr_p1'],
    'p2': lat_ds['v_corr_p2'],
    'delta': lat_ds['v_corr_delta'],
}
lat_omega = {
    'p1': lat_ds['omega_corr_p1'],
    'p2': lat_ds['omega_corr_p2'],
    'delta': lat_ds['omega_corr_delta'],
}

print('lon cache vars:', list(lon_ds.data_vars))
print('lat cache vars:', list(lat_ds.data_vars))

# 3. Longitude-Pressure: q Shading

In [ ]:
for period, title, label, out_name in [
    ('p1', 'Longitude-pressure q correlation with Niño3.4 - P1 1981-2006', 'q correlation with Niño3.4 (r)', 'corr_xsec_lon_q_p1.png'),
    ('p2', 'Longitude-pressure q correlation with Niño3.4 - P2 2007-2025', 'q correlation with Niño3.4 (r)', 'corr_xsec_lon_q_p2.png'),
    ('delta', 'Longitude-pressure q correlation with Niño3.4 - P2-P1', 'q correlation difference with Niño3.4 (Δr)', 'corr_xsec_lon_q_delta.png'),
]:
    render_cross_section(
        lon_q[period],
        lon_u[period],
        lon_omega[period],
        x_dim='lon',
        title=title,
        x_label='Longitude (°E)',
        x_ticks=LON_TICKS,
        cbar_label=label,
        out_name=out_name,
        vector_label='vector components: u correlation and -omega correlation',
    )


# 4. Latitude-Pressure: q Shading

In [ ]:
for period, title, label, out_name in [
    ('p1', 'Latitude-pressure q correlation with Niño3.4 - P1 1981-2006', 'q correlation with Niño3.4 (r)', 'corr_xsec_lat_q_p1.png'),
    ('p2', 'Latitude-pressure q correlation with Niño3.4 - P2 2007-2025', 'q correlation with Niño3.4 (r)', 'corr_xsec_lat_q_p2.png'),
    ('delta', 'Latitude-pressure q correlation with Niño3.4 - P2-P1', 'q correlation difference with Niño3.4 (Δr)', 'corr_xsec_lat_q_delta.png'),
]:
    render_cross_section(
        lat_q[period],
        lat_v[period],
        lat_omega[period],
        x_dim='lat',
        title=title,
        x_label='Latitude (°N)',
        x_ticks=LAT_TICKS,
        cbar_label=label,
        out_name=out_name,
        vector_label='vector components: v correlation and -omega correlation',
    )


# 5. Longitude-Pressure: w Shading

In [ ]:
for period, title, label, out_name in [
    ('p1', 'Longitude-pressure omega correlation with Niño3.4 - P1 1981-2006', 'omega correlation with Niño3.4 (r)', 'corr_xsec_lon_w_p1.png'),
    ('p2', 'Longitude-pressure omega correlation with Niño3.4 - P2 2007-2025', 'omega correlation with Niño3.4 (r)', 'corr_xsec_lon_w_p2.png'),
    ('delta', 'Longitude-pressure omega correlation with Niño3.4 - P2-P1', 'omega correlation difference with Niño3.4 (Δr)', 'corr_xsec_lon_w_delta.png'),
]:
    render_cross_section(
        lon_omega[period],
        lon_u[period],
        lon_omega[period],
        x_dim='lon',
        title=title,
        x_label='Longitude (°E)',
        x_ticks=LON_TICKS,
        cbar_label=label,
        out_name=out_name,
        vector_label='vector components: u correlation and -omega correlation',
    )


# 6. Latitude-Pressure: w Shading

In [ ]:
for period, title, label, out_name in [
    ('p1', 'Latitude-pressure omega correlation with Niño3.4 - P1 1981-2006', 'omega correlation with Niño3.4 (r)', 'corr_xsec_lat_w_p1.png'),
    ('p2', 'Latitude-pressure omega correlation with Niño3.4 - P2 2007-2025', 'omega correlation with Niño3.4 (r)', 'corr_xsec_lat_w_p2.png'),
    ('delta', 'Latitude-pressure omega correlation with Niño3.4 - P2-P1', 'omega correlation difference with Niño3.4 (Δr)', 'corr_xsec_lat_w_delta.png'),
]:
    render_cross_section(
        lat_omega[period],
        lat_v[period],
        lat_omega[period],
        x_dim='lat',
        title=title,
        x_label='Latitude (°N)',
        x_ticks=LAT_TICKS,
        cbar_label=label,
        out_name=out_name,
        vector_label='vector components: v correlation and -omega correlation',
    )


# 7. Revision Summary

- Plot limits are now forced to `95E-125E` for longitude-pressure sections and `10S-5N` for latitude-pressure sections.
- Titles, axes, colorbars, and annotations explicitly label fields as correlations with Niño3.4.
- Quivers are labeled as correlation-component diagnostics, not actual circulation vectors.
- Panels with all-NaN omega skip quiver and show a visible omega-unavailable annotation instead of failing silently.
